### 미니프로젝트1
- 공공데이터 기반 지역별 통계 분석

In [357]:
#상대 경로 - 파일 열고 구조 확인 후 변수에 넣기
import pandas as pd
import re


In [358]:
# 파일읽기
toilet = pd.read_csv(r'..\data\raw\공중화장실정보.csv', encoding= 'cp949')
pop = pd.read_csv(r'..\data\raw\주민등록인구수_20260630.csv', encoding= 'cp949')
print("화장실 원본:", toilet.shape)
print("인구 원본:", pop.shape)


화장실 원본: (53552, 34)
인구 원본: (3618, 230)


In [359]:
# ===================================================
# 2. 화장실 데이터 전처리
# ===================================================


In [360]:
def parse_region(row):
    """
    도로명주소와 지번주소를 이용하여
    sido, sigungu를 추출한다.
    """

    # 도로명주소를 먼저 사용하고,
    # 실패하면 지번주소를 사용
    addresses = [
        row.get('소재지도로명주소'),
        row.get('소재지지번주소')
    ]

    for addr in addresses:

        if pd.isna(addr):
            continue

        addr = str(addr).strip()

        if not addr:
            continue

        # --------------------------------
        # 1. 시/구가 붙어 있는 경우 띄어쓰기 보정
        # 예:
        # 수원시영통구 → 수원시 영통구
        # --------------------------------
        addr = re.sub(
            r'([가-힣]+시)([가-힣]+구)(?=\s|$)',
            r'\1 \2',
            addr
        )

        tokens = addr.split()

        if len(tokens) < 2:
            continue

        sido = tokens[0]
        second = tokens[1]

        # --------------------------------
        # 2. 정상적인 시군구 형태
        #
        # 서울특별시 종로구
        # 경기도 수원시 영통구
        # 강원특별자치도 강릉시
        # --------------------------------
        if re.fullmatch(r'[가-힣]+[시군구]', second):

            sigungu = second

            # 수원시 영통구처럼
            # '시 + 구' 형태인 경우
            if (
                second.endswith('시')
                and len(tokens) >= 3
                and re.fullmatch(r'[가-힣]+구', tokens[2])
            ):
                sigungu = f'{second} {tokens[2]}'

            return sido, sigungu

        # --------------------------------
        # 3. 시군구와 도로명이 붙어 있는 경우
        #
        # 서울특별시 용산구녹사평대로11길
        # 서울특별시 성동구고산자로
        #
        # 주의:
        # '압구정로' → '압구'가 되면 안 됨
        # --------------------------------

        # 실제 행정구역명 뒤에 도로명/주소가 이어지는 경우만
        # '시/군/구'를 추출한다.
        match = re.match(
            r'^(.+?(?:특별시|광역시|특별자치시|특별자치도|시|군|구))'
            r'(?=[가-힣]+(?:대로|로|길|거리)\b)',
            second
        )

        if match:
            sigungu = match.group(1)

            # 혹시 시 뒤에 구가 붙은 형태
            # 수원시영통구청사로...
            if sigungu.endswith('시'):

                rest = second[len(sigungu):]

                match_gu = re.match(
                    r'^([가-힣]+구)'
                    r'(?=[가-힣]+(?:대로|로|길|거리)\b)',
                    rest
                )

                if match_gu:
                    sigungu = (
                        f'{sigungu} {match_gu.group(1)}'
                    )

            return sido, sigungu

    # --------------------------------
    # 4. 두 주소 모두 파싱 실패
    # --------------------------------
    return None, None

In [361]:
toilet[['sido', 'sigungu']] = toilet.apply(
    parse_region,
    axis=1,
    result_type='expand'
)

In [362]:
# 결측 수정 1) 수동 보정 (오탈자 등)
manual_fix = {
    '과쳔시': '과천시',
    '봉하군': '봉화군',
    '주시 덕진구': '전주시 덕진구',
    '포항시 부구': '포항시 북구',
    '수원특례시 권선구': '수원시 권선구',
    '서울특별시 송파구':'송파구'
}

toilet['sigungu'] = toilet['sigungu'].replace(manual_fix)  

In [363]:
# 결측 수정 2) 인천 신설구 → 옛 구명 역매핑 (2026.06 기준 데이터 정합성 맞추기)
incheon_reverse_map = {
    '영종구': '중구',
    '제물포구': '중구',
    '검단구': '서구',
    '서해구': '서구'
}

toilet['sigungu'] = toilet['sigungu'].replace(
    incheon_reverse_map
)

In [364]:
# 구 정보 없이 등록된 행 -결측처리
remaining = toilet[toilet['sigungu'].isin(['포항시','부천시','중구 중구'])]
print(remaining.shape[0], "건")
print(remaining[['소재지도로명주소','소재지지번주소']])

6 건
                   소재지도로명주소                     소재지지번주소
16996   경기도 부천시 경인로 36(송내동)                         NaN
17089   경기도 부천시 계남로 219(중동)                         NaN
17094  경기도 부천시 옥길로 143(옥길동)                         NaN
33880       경상북도 포항시 장량로 56      경상북도 포항시 북구 장성동 산97-13
33970      경상북도 포항시 양학로 166       경상북도 포항시 북구 학잠동 216-1
34051                   NaN  경상북도 포항시 남구호미곶면 구만2리 175-6


In [ ]:
# 파싱 결과 확인
print("sido 파싱 실패:", toilet['sido'].isna().sum())
print("sigungu 파싱 실패:", toilet['sigungu'].isna().sum())

# 수원시 관련 확인
print(
    toilet[
        toilet['소재지도로명주소'].astype(str).str.contains(
            '수원', na=False
        )
    ][
        ['소재지도로명주소', '소재지지번주소', 'sido', 'sigungu']
    ]
    .head(20)
    .to_string(index=False)
)

sido 파싱 실패: 2151
sigungu 파싱 실패: 2151
                                 소재지도로명주소                          소재지지번주소  sido sigungu
부산광역시 해운대구 해운대해변로265번길 6, 해운대우체국및연수원 (중동) 부산광역시 해운대구 중동 1391-72 해운대우체국및연수원 부산광역시    해운대구
                        부산광역시 금정구 수원지로 58                부산광역시 금정구 회동동 499 부산광역시     금정구
                     대전광역시 서구 가수원중로 43-11                              NaN 대전광역시      서구
                       대전광역시 서구 가수원로 91-9                              NaN 대전광역시      서구
                    대전광역시 서구 가수원중로65번길 22                              NaN 대전광역시      서구
                        대전광역시 서구 가수원로 105                              NaN 대전광역시      서구
                      대전광역시 서구 가수원로 91-11                              NaN 대전광역시      서구
                        대전광역시 서구 가수원로 109                              NaN 대전광역시      서구
          대전광역시 서구 계백로 1176, 쌍용주유소 (가수원동)        대전광역시 서구 가수원동 210-7 쌍용주유소 대전광역시      서구
                    대전광역시 서구 가수원중로65번길 22                              Na

In [ ]:
#압구정 (구로 잘라서 압구.로 잘리는 이슈)
print(
    toilet[
        toilet['소재지도로명주소'].astype(str).str.contains(
            '압구정', na=False
        )
    ][
        ['소재지도로명주소', 'sido', 'sigungu']
    ]
    .head(20)
    .to_string(index=False)
)

                 소재지도로명주소  sido sigungu
     서울특별시 압구정로 306 (신사동) 서울특별시      압구
서울특별시 강남구 압구정로 지하172(신사동) 서울특별시     강남구
  서울특별시 압구정로29길 68 (압구정동)  None    None
     서울특별시 압구정로 108 (신사동) 서울특별시      압구
     서울특별시 압구정로 154 (신사동) 서울특별시      압구
  서울특별시 압구정로29길 71 (압구정동)  None    None
        서울특별시 강남구 압구정동422 서울특별시     강남구
       서울특별시 강남구 압구정로 311 서울특별시     강남구
     서울특별시 강남구 압구정로 38길 7 서울특별시     강남구
     서울특별시 강남구 압구정로33길 48 서울특별시     강남구
       서울특별시 강남구 압구정로 161 서울특별시     강남구
  서울특별시 압구정로29길 68 (압구정동)  None    None
    서울특별시 압구정로 201 (압구정동) 서울특별시      압구
     서울특별시 압구정로 302 (신사동) 서울특별시      압구
       서울특별시 강남구 압구정로 128 서울특별시     강남구
 서울특별시 압구정로 113-22 (압구정동) 서울특별시      압구
  서울특별시 압구정로29길 69 (압구정동)  None    None
     서울특별시 압구정로 152 (신사동) 서울특별시      압구


In [367]:
print(
    "sido 파싱 실패:",
    toilet['sido'].isna().sum()
)

print(
    "sigungu 파싱 실패:",
    toilet['sigungu'].isna().sum()
)

sido 파싱 실패: 2151
sigungu 파싱 실패: 2151


In [368]:
# print(
#     toilet[
#         toilet['sigungu'].astype(str).str.contains(
#             '수원',
#             na=False
#         )
#     ][['sido', 'sigungu']]
#     .drop_duplicates()
# )

In [369]:
# 2-2. 변기수 파생 컬럼 (성인/장애인/어린이 구분해서 저장)
toilet['male_seats'] = (
    toilet['남성용-대변기수'] + toilet['남성용-소변기수']
)
toilet['female_seats'] = toilet['여성용-대변기수']

toilet['disabled_seats'] = (
    toilet['남성용-장애인용대변기수'] + toilet['남성용-장애인용소변기수'] +
    toilet['여성용-장애인용대변기수']
)
toilet['child_seats'] = (
    toilet['남성용-어린이용대변기수'] + toilet['남성용-어린이용소변기수'] +
    toilet['여성용-어린이용대변기수']
)
toilet['total_seats'] = (
    toilet['male_seats'] + toilet['female_seats'] +
    toilet['disabled_seats'] + toilet['child_seats']
)

In [370]:
# 2-3. 기저귀교환대 Y/N → 0/1
toilet['has_diaper_table'] = (toilet['기저귀교환대유무'] == 'Y').astype(int)

In [ ]:
#관리번호 toilet id 합산 
toilet_clean = toilet[[
    '관리번호',
    'sido',
    'sigungu',
    'male_seats',
    'female_seats',
    'disabled_seats',
    'child_seats',
    'total_seats',
    'has_diaper_table'
]].rename(columns={'관리번호': 'toilet_id'})

In [ ]:
#결측 or 중복 확인
print(
    "sido 결측:",
    toilet_clean['sido'].isna().sum()
)

print(
    "sigungu 결측:",
    toilet_clean['sigungu'].isna().sum()
)

print(
    "toilet_id 중복:",
    toilet_clean['toilet_id'].duplicated().sum()
)

# 시도/시군구가 없는 행 제거
toilet_clean = toilet_clean.dropna(
    subset=['sido', 'sigungu']
)

# 동일한 화장실 ID가 있으면 첫 번째만 유지
toilet_clean = toilet_clean.drop_duplicates(
    subset=['toilet_id'],
    keep='first'
)

print(
    "최종 화장실 데이터:",
    toilet_clean.shape
)

sido 결측: 2151
sigungu 결측: 2151
toilet_id 중복: 0


최종 화장실 데이터: (51401, 9)


In [ ]:
#합산하지 못한 주소들 확인
failed = toilet[toilet['sigungu'].isna()]

print(
    failed[['소재지도로명주소', '소재지지번주소']]
    .head(30)
    .to_string(index=False)
)

                           소재지도로명주소                   소재지지번주소
               서울특별시 용산구녹사평대로11길 24                       NaN
                    서울특별시 용산구원효로178                       NaN
                    서울특별시 면목동 168-1                       NaN
서울특별시 사가정로52길 13, 5층(면목7동주민센터 임시청사)                       NaN
                  서울특별시 동일로 114길 10                       NaN
       서울특별시 목동중앙로9길 40, 용왕산 유아숲체험원                       NaN
       서울특별시 목동중앙로9길 40, 용왕산 유아숲체험원                       NaN
       서울특별시 목동중앙로9길 40, 용왕산 유아숲체험원                       NaN
                    서울특별시 남부순환로 823                       NaN
         영등포구 당산로 123(당산동3가, 영등포구청)    영등포구 당산동3가 385-1 영등포구청
                영등포구 당산로 123(당산동3가) 영등포구 당산동3가 385-1 영등포구 보건소
                       영등포구 경인로 843                 영등포동3가 33
     영등포구 선유동1로 80(당산동3가, 영등포구청 별관)   영등포구 당산동3가 560 영등포구청 별관
                  서울특별시 영등포동 3가 441                       NaN
                  서울특별시 강남대로146길 28                       NaN
        

In [374]:
# ===================================================
# 3. 인구 데이터 전처리 (읍면동 → 시군구 단위로 집계)
# ===================================================

In [ ]:
#나잇대별로 인원수 확인 
male_cols = [
    c for c in pop.columns
    if c.endswith('세남자')
    or c == '100세이상 남자'
]

female_cols = [
    c for c in pop.columns
    if c.endswith('세여자')
    or c == '100세이상 여자'
]

elderly_male_cols = [
    c for c in male_cols
    if any(str(a) in c for a in range(65, 111))
]

elderly_female_cols = [
    c for c in female_cols
    if any(str(a) in c for a in range(65, 111))
]

child_male_cols = [
    f'{a}세남자'
    for a in range(0, 7)
]

child_female_cols = [
    f'{a}세여자'
    for a in range(0, 7)
]

pop['elderly_pop'] = (
    pop[elderly_male_cols].sum(axis=1)
    + pop[elderly_female_cols].sum(axis=1)
)

pop['child_pop'] = (
    pop[child_male_cols].sum(axis=1)
    + pop[child_female_cols].sum(axis=1)
)

C:\Users\CY\AppData\Local\Temp\ipykernel_11404\3389492074.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pop['elderly_pop'] = (
C:\Users\CY\AppData\Local\Temp\ipykernel_11404\3389492074.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pop['child_pop'] = (


In [376]:
# 3-2. 읍면동 단위 → 시군구 단위로 집계 (세종시 NaN 처리 추가)
pop['시군구명'] = pop['시군구명'].astype(str).str.strip()
pop['시군구명'] = pop['시군구명'].replace('nan', '')  # astype(str)이 NaN을 'nan' 문자열로 바꿔버리므로 다시 정리   

population_clean = pop.groupby(['시도명', '시군구명']).agg(
    male_pop=('남자', 'sum'),
    female_pop=('여자', 'sum'),
    total_pop=('계', 'sum'),
    elderly_pop=('elderly_pop', 'sum'),
    child_pop=('child_pop', 'sum')
).reset_index().rename(columns={'시도명': 'sido', '시군구명': 'sigungu'})

In [377]:

# ===================================================
# 4. 매칭 검증 (JOIN 전에 반드시 확인)
# ===================================================

In [ ]:
#4-1. 두 데이터간 매칭 확인.
toilet_region = (
    toilet_clean[['sido', 'sigungu']]
    .drop_duplicates()
)

pop_region = (
    population_clean[['sido', 'sigungu']]
    .drop_duplicates()
)

region_check = toilet_region.merge(
    pop_region,
    on=['sido', 'sigungu'],
    how='left',
    indicator=True
)

print(
    "지역 매칭률:",
    (region_check['_merge'] == 'both').mean()
)

print("인구 데이터에 없는 화장실 지역:")
print(
    region_check[
        region_check['_merge'] == 'left_only'
    ][['sido', 'sigungu']]
)

지역 매칭률: 0.7907801418439716
인구 데이터에 없는 화장실 지역:
          sido sigungu
5        서을특별시     광진구
21          서울    영등포구
26       서울특별시      압구
71          울산     울주군
76         경기도     수원시
77      경기도수원시     영통구
78      경기도수원시     권선구
79         경기도     영통구
80         경기도     팔달구
81         경기도     화성시
85         경기도     송파구
86         경기도     중원구
87         경기도     성남시
91         경기도     안양시
95         경기도     부천시
101        경기도     안산시
140       충청북도      모시
143         충북     보은군
166    전북특별자치도     덕진구
167    전북특별자치도     전주시
168    전북특별자치도     완산구
184       경상북도     포항시
192         경북     상주시
206         경북     봉화군
207       경상붓도     봉화군
208        경상북     봉화군
217         경남     양산시
219       경상님도     양산시
236         충남     계룡시
244       경상남도     창원시
246    세종특별자치시     금남구
247    세종특별자치시    국책연구
253  전남광주통합특별시     목포시
254  전남광주통합특별시     여수시
255  전남광주통합특별시     순천시
256  전남광주통합특별시     나주시
257  전남광주통합특별시     광양시
258  전남광주통합특별시      동구
259  전남광주통합특별시      서구
260  전남광주통합특별시      남구
261  전남광주통합

In [379]:
toilet_clean['toilet_id'] = toilet_clean['toilet_id'].astype(str)
print("toilet_id 최대 길이:",
      toilet_clean['toilet_id'].str.len().max())

toilet_id 최대 길이: 18


In [380]:
toilet_sigungu = set(toilet_clean['sigungu'])
pop_sigungu = set(population_clean['sigungu'])

print("매칭률:", toilet_clean['sigungu'].isin(pop_sigungu).mean())
print("화장실에만 있는 지역명:", list(toilet_sigungu - pop_sigungu)[:15])

매칭률: 0.9985019746697535
화장실에만 있는 지역명: ['권선구', '금남구', '국책연구', '팔달구', '덕진구', '완산구', '모시', '화성시', '안산시', '중원구', '영통구', '성남시', '압구', '수원시', '부천시']


In [381]:
unmatched = toilet_clean[
    ~toilet_clean['sigungu'].isin(population_clean['sigungu'])
]

print("FK 매칭 실패:", len(unmatched))
print(unmatched[['toilet_id', 'sigungu']].head())

FK 매칭 실패: 77
               toilet_id sigungu
4734  202232200000100362      압구
4753  202232200000100349      압구
4778  202232200000100353      압구
4934  202232200000100371      압구
4950  202232200000100361      압구


In [382]:
# (sido, sigungu) 복합키 기준 매칭 검증
toilet_pairs = set(zip(toilet_clean['sido'], toilet_clean['sigungu']))
pop_pairs = set(zip(population_clean['sido'], population_clean['sigungu']))

unmatched = toilet_pairs - pop_pairs
print(f"매칭 안 되는 (sido, sigungu) 조합: {len(unmatched)}개")
print(list(unmatched)[:20])

매칭 안 되는 (sido, sigungu) 조합: 59개
[('전북특별자치도', '전주시'), ('전남광주통합특별시', '목포시'), ('전북특별자치도', '완산구'), ('전남광주통합특별시', '영광군'), ('서울', '영등포구'), ('전남광주통합특별시', '장성군'), ('충북', '보은군'), ('전남광주통합특별시', '북구'), ('전남광주통합특별시', '담양군'), ('경기도', '화성시'), ('전남광주통합특별시', '광산구'), ('경북', '상주시'), ('경상님도', '양산시'), ('전남광주통합특별시', '신안군'), ('경상남도', '창원시'), ('전남광주통합특별시', '완도군'), ('울산', '울주군'), ('경기도', '안산시'), ('경기도', '중원구'), ('전남광주통합특별시', '남구')]


In [383]:
# ---- ① 광주-전남 통합 역매핑 ----
gwangju_gu = {'동구', '서구', '남구', '북구', '광산구'}

def fix_gwangju_jeonnam(row):
    if row['sido'] != '전남광주통합특별시':
        return row['sido']
    return '광주광역시' if row['sigungu'] in gwangju_gu else '전라남도'

toilet_clean['sido'] = toilet_clean.apply(fix_gwangju_jeonnam, axis=1)

# ---- ② sido 축약형 + ③ 오탈자 보정 ----
sido_fix = {
    '서울': '서울특별시',
    '충북': '충청북도',
    '경북': '경상북도',
    '울산': '울산광역시',
    '경상님도': '경상남도',   # 오탈자
}
toilet_clean['sido'] = toilet_clean['sido'].replace(sido_fix)

# ---- 재검증 ----
toilet_pairs = set(zip(toilet_clean['sido'], toilet_clean['sigungu']))
pop_pairs = set(zip(population_clean['sido'], population_clean['sigungu']))
unmatched = toilet_pairs - pop_pairs
print(f"남은 불일치: {len(unmatched)}개")
print(list(unmatched))

남은 불일치: 26개
[('전북특별자치도', '전주시'), ('전북특별자치도', '완산구'), ('경기도', '화성시'), ('경상남도', '창원시'), ('경기도', '안산시'), ('경기도', '중원구'), ('경상북도', '포항시'), ('경기도수원시', '영통구'), ('경상북', '봉화군'), ('경기도', '영통구'), ('경상붓도', '봉화군'), ('세종특별자치시', '금남구'), ('경기도', '송파구'), ('경기도', '성남시'), ('충남', '계룡시'), ('경기도', '수원시'), ('서을특별시', '광진구'), ('경기도', '안양시'), ('경기도', '부천시'), ('세종특별자치시', '국책연구'), ('충청북도', '모시'), ('경기도수원시', '권선구'), ('경남', '양산시'), ('경기도', '팔달구'), ('전북특별자치도', '덕진구'), ('서울특별시', '압구')]


In [384]:
# ---- 세종특별자치시: 구 단위가 없으므로 sigungu 강제로 빈 값 처리 ----
toilet_clean.loc[toilet_clean['sido'] == '세종특별자치시', 'sigungu'] = ''

# ---- 수원시: '경기도수원시'로 붙어쓴 경우 분리 ----
toilet_clean.loc[toilet_clean['sido'] == '경기도수원시', 'sido'] = '경기도'

# ---- 수원시 4개 구: 시 이름 없이 구만 있는 경우 '수원시 ' 접두어 추가 ----
suwon_gu = {'영통구', '권선구', '팔달구', '장안구'}
mask = (toilet_clean['sido'] == '경기도') & (toilet_clean['sigungu'].isin(suwon_gu))
toilet_clean.loc[mask, 'sigungu'] = '수원시 ' + toilet_clean.loc[mask, 'sigungu']

# ---- 송파구: "경기도 서울특별시 송파구" 오류 주소 보정 ----
toilet_clean.loc[toilet_clean['sigungu'] == '송파구', 'sido'] = '서울특별시'

# ---- sido 축약/오탈자 보정 (추가분 포함) ----
sido_fix = {
    '서울': '서울특별시', '서을특별시': '서울특별시',
    '충북': '충청북도', '충남': '충청남도',
    '경북': '경상북도', '경상북': '경상북도', '경상붓도': '경상북도',
    '경남': '경상남도', '경상님도': '경상남도',
    '울산': '울산광역시',
}
toilet_clean['sido'] = toilet_clean['sido'].replace(sido_fix)

# ---- 재검증 ----
toilet_pairs = set(zip(toilet_clean['sido'], toilet_clean['sigungu']))
pop_pairs = set(zip(population_clean['sido'], population_clean['sigungu']))
unmatched = toilet_pairs - pop_pairs
print(f"남은 불일치: {len(unmatched)}개")
print(list(unmatched))

# ---- 여전히 안 맞는 행 최종 제외 (불완전 주소) ----
before_n = toilet_clean.shape[0]
toilet_clean = toilet_clean[
    toilet_clean.apply(lambda r: (r['sido'], r['sigungu']) in pop_pairs, axis=1)
].copy()
print(f"최종 제외: {before_n - toilet_clean.shape[0]}건")
print(f"최종 toilet_clean: {toilet_clean.shape}")

남은 불일치: 14개
[('충청북도', '모시'), ('경기도', '화성시'), ('전북특별자치도', '전주시'), ('경기도', '성남시'), ('전북특별자치도', '완산구'), ('경상남도', '창원시'), ('전북특별자치도', '덕진구'), ('경기도', '수원시'), ('경기도', '안산시'), ('경기도', '중원구'), ('서울특별시', '압구'), ('경기도', '안양시'), ('경상북도', '포항시'), ('경기도', '부천시')]
최종 제외: 69건
최종 toilet_clean: (51332, 9)


In [385]:
dup_check = population_clean[
    population_clean.duplicated(subset=['sido','sigungu'], keep=False)
]
print(dup_check)

Empty DataFrame
Columns: [sido, sigungu, male_pop, female_pop, total_pop, elderly_pop, child_pop]
Index: []


In [386]:
#========================
# 적재
#========================

In [387]:
import pymysql
import os

from dotenv import load_dotenv
load_dotenv()



conn = pymysql.connect(
    host=os.environ.get('DB_HOST','127.0.0.1'), # 데이터베이스 서버(컴퓨터) 주소 (localhost, 127.0.0.1)
    port=int(os.environ.get('DB_PORT', '3306')),
    user=os.environ.get('DB_USER', 'analyst'),
    password=os.environ.get('DB_PASSWORD', ''),
    database=os.environ.get('DB_NAME', 'toilet_db'),
    charset='utf8mb4'      
)
cur = conn.cursor()

In [388]:
pop_records = population_clean[[
    'sido',
    'sigungu',
    'male_pop',
    'female_pop',
    'total_pop',
    'elderly_pop',
    'child_pop'
]].values.tolist()

cur.executemany("""
    INSERT INTO tb_population (
        sido,
        sigungu,
        male_pop,
        female_pop,
        total_pop,
        elderly_pop,
        child_pop
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s)
""", pop_records)

conn.commit()

print(f"tb_population {len(pop_records)}건 적재 완료")

tb_population 255건 적재 완료


In [389]:

# =========================
# 2. tb_toilet 적재
# =========================

toilet_records = toilet_clean[[
    'toilet_id',
    'sido',
    'sigungu',
    'male_seats',
    'female_seats',
    'disabled_seats',
    'child_seats',
    'total_seats',
    'has_diaper_table'
]].values.tolist()

try:
    cur.executemany("""
        INSERT INTO tb_toilet (
            toilet_id,
            sido,
            sigungu,
            male_seats,
            female_seats,
            disabled_seats,
            child_seats,
            total_seats,
            has_diaper_table
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, toilet_records)

    conn.commit()

    print(f"tb_toilet {len(toilet_records)}건 적재 완료")

except pymysql.IntegrityError as e:
    print("무결성 제약 위반:", e)
    conn.rollback()



tb_toilet 51332건 적재 완료


In [390]:
# =========================
# 3. 적재 검증
# =========================

cur.execute("SELECT COUNT(*) FROM tb_population")
pop_db_count = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM tb_toilet")
toilet_db_count = cur.fetchone()[0]

print("\n===== 적재 검증 =====")
print(f"population 원본 행 수 : {len(population_clean)}")
print(f"population DB 행 수   : {pop_db_count}")

print(f"toilet 원본 행 수     : {len(toilet_clean)}")
print(f"toilet DB 행 수       : {toilet_db_count}")


# 샘플 조회
cur.execute("""
    SELECT *
    FROM tb_population
    LIMIT 5
""")

print("\n[tb_population 샘플]")
for row in cur.fetchall():
    print(row)


cur.execute("""
    SELECT *
    FROM tb_toilet
    LIMIT 5
""")

print("\n[tb_toilet 샘플]")
for row in cur.fetchall():
    print(row)


cur.close()
conn.close()


===== 적재 검증 =====
population 원본 행 수 : 255
population DB 행 수   : 255
toilet 원본 행 수     : 51332
toilet DB 행 수       : 51332

[tb_population 샘플]
('강원특별자치도', '강릉시', 101963, 103703, 205666, 58369, 6342)
('강원특별자치도', '고성군', 13936, 13268, 27204, 9753, 810)
('강원특별자치도', '동해시', 43362, 42227, 85589, 23432, 2810)
('강원특별자치도', '삼척시', 30872, 29422, 60294, 19952, 1734)
('강원특별자치도', '속초시', 38976, 39917, 78893, 21146, 2571)

[tb_toilet 샘플]
('199644300000100001', '충청북도', '옥천군', 3, 2, 0, 0, 5, 0)
('201544300000100001', '충청북도', '옥천군', 2, 1, 0, 0, 3, 0)
('202030700000100182', '서울특별시', '성북구', 3, 1, 0, 0, 4, 0)
('202030700000100183', '서울특별시', '성북구', 2, 2, 0, 0, 4, 0)
('202030700000100184', '서울특별시', '성북구', 2, 1, 0, 0, 3, 0)


In [391]:
print(
    "복합키 중복:",
    population_clean.duplicated(
        subset=['sido', 'sigungu']
    ).sum()
)

복합키 중복: 0


In [392]:
print(
    population_clean.duplicated(
        subset=['sido', 'sigungu']
    ).sum()
)

0
